Key modules:
- pyspark.sql.SparkSession -> spark object used for creating and reading DataFrames
- pyspark.sql.types -> StructType, StructField, and PySpark-specific data types for schemas
- pyspark.sql.functions -> col(), expr()

In [ ]:
import os

os.environ["JAVA_HOME"] = "/Users/jerry/.sdkman/candidates/java/17.0.13-tem"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:{os.environ["PATH"]}'

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("operations")
    .master("local[*]")
    .getOrCreate()
)

### Schemas
Spark DataFrames are tabular at the top level, so a DataFrame schema describes the top-level column names as well as types. A DataFrame schema object (`df.schema`) is a `StructType` object.

Beyond all of the scalar / simple data types, there are 3 nested types:
- `ArrayType`: an ordered list of elements of the same type
- `MapType`: an unordered map of key-value pairs where keys are dynamic (JSON schema inference produces `StructType`, not `MapType`)
- `StructType`: a list or tuple of `StructField` data types (imagine a further-nested object schema)

In [ ]:
# Infer and print a schema
df = (
    spark.read
    .format("json")
    .load("data/flight-data/json/2015-summary.json") # must be last; transforms builder (DataFrameReader) into DataFrame
)

df.printSchema()

In [ ]:
# Define a custom schema (useful for prod ETL pipelines)
from pyspark.sql.types import StructField, StructType, StringType, LongType

flight_schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", LongType(), True)
])

df = (
    spark.read
    .format("json")
    .schema(flight_schema)
    .load("data/flight-data/json/2015-summary.json")
)

df.printSchema()

### Columns, Rows, and DataFrame Creation

In [ ]:
# Creating Column references and Rows
from pyspark.sql.functions import col
from pyspark.sql import Row

# Both col("name") and df["name"] produce the same Column expression
# However using the col() function creates a free column reference (resolved against a particular DataFrame) while df["name"] asks that particular DataFrame object for its column
# It's better to use col("name") most of the time as bracket notation is overloaded
print(df.select(col("count")))
print(df.select(df["count"]))

# To create a Row, you can either supply kwargs (key is col name) or generate a factory with specified column names
# Values within a row can be accessed by indexing position -- row[1] -- or by field name -- row["name"]
row1 = Row(id=1, cost=100.0, name="Zoe")
RowCreator = Row("id", "cost", "name")
row2 = RowCreator(2, 50.1, "Isabella")
print(row1)
print(row2)

In [ ]:
# To create a DataFrame manually (not from a data source), we provide an iterable of rows/tuples and a schema
# The names of row fields 
from pyspark.sql.types import StructType, StructField, TimestampType, DecimalType, StringType
from datetime import datetime
from decimal import Decimal

stock_schema = StructType([
    StructField("Timestamp", TimestampType(), False),
    StructField("Price", DecimalType(10, 2), False),
    StructField("Ticker", StringType(), False)
])

# The values are matched by position order
# The schema provided overwrites any name mismatches in individual rows
StockRow = Row("Timestamp", "Price", "StockName")
row1 = StockRow(datetime(2020, 10, 1, 14, 30, 0), Decimal("50.25"), "APPL")
row2 = StockRow(datetime(2020, 10, 1, 15, 0, 0), Decimal("56.78"), "APPL")

stock_df = spark.createDataFrame(
    data=[row1, row2],
    schema=stock_schema
)

print(stock_df.columns) # ['Timestamp', 'Price', 'Ticker'] <- "StockName" row field was overwritten
stock_df.show()

In [ ]:
# Adding a Column to a DataFrame
from pyspark.sql.functions import lit, expr

# Adding a constant value to a DataFrame
new_stock_df = stock_df.withColumn(
    "Industry",
    lit("Technology") # Place any literal value in here for a new column of constant values
)

# Adding a column based on another column
new_stock_df = new_stock_df.withColumn(
    "IsExpensive",
    expr("Price > 50")
)

new_stock_df.show()

In [ ]:
# Removing a Column from a DataFrame
new_stock_df.drop("IsExpensive").show() # Note that this still returns a new DataFrame
new_stock_df.show()

In [ ]:
# .withColumn() can be used to add a new Column, but if the column name provided already exists, it replaces it
from pyspark.sql.functions import expr, col
from pyspark.sql.types import StringType

new_stock_df = new_stock_df.withColumn("IsExpensive", expr("Price > 100"))
new_stock_df.show()

# A cast is done using a column reference of the original with .cast() method
new_stock_df.withColumn("IsExpensive", col("IsExpensive").cast(StringType())).printSchema()

### Selecting Data From DataFrames
`.select()` and `.selectExpr()` allow for the equivalent of SQL queries in PySpark. Both return new `DataFrame`s by choosing certain columns or computed expressions.

In [ ]:
# .select() is used when we are choosing or outputting columns from a DataFrame
# When we don't need excess transformations, we can use the string of the column name directly
stock_df.select("Price").show(1)
print("\n=============\n")
stock_df.select("Timestamp", "Price").show() # equivalent to SELECT Timestamp, Price FROM stock_df

In [ ]:
# When we do need to transform the output, prefer passing Column references
stock_df.select(
    col("Timestamp"),
    col("Price") + 2.50
).show()

In [ ]:
# We can also select certain expressions directly
# An expression in PySpark is a "description of how to compute a column value"
# expr() returns a Column type describing transformations on a Column
from pyspark.sql.functions import expr

stock_df.select(
    col("Timestamp"), # All columns must be either strings or Column types
    expr("Price + 2.50 as IncreasedPrice")
).show()

In [ ]:
# .selectExpr() is a DataFrame method that is equivalent to .select(expr(...), expr(...))
# It takes in a comma-separated list of strings that are parsed as SQL
stock_df.selectExpr(
    "Timestamp",
    "Price + 2.50 as IncreasedPrice",
    "CONCAT(Ticker, ' Stock') as SecurityType"
).show()

### Filtering by Condition
Both `df.where()` and `df.filter()` work the same way in PySpark: to return a new DataFrame containing only the rows where an expression is evaluated to True.

In [ ]:
# Both take in either a SQL expression string or a boolean Column expression, both needing to evaluate to True/False
# The Column must evaluate to a boolean (either the column reference itself contains rows of booleans or it is used in an expression outputting a boolean result)
# Values can be null; nulls are skipped after filtering step so treated same as false
stock_df.where("Price > 55.0").show()
stock_df.where(col("Price") > 55).show()

In [ ]:
# Obtain rows with distinct values in a particular column
# .distinct() is a filter keeping only unique rows
stock_df.select("Ticker").distinct().count() # .count() returns number of rows in DataFrame

### Sampling and Splitting Data

In [ ]:
# Use original flight data 2015 summary
# df.sample(with_replacement: bool, fraction: float, seed: int)
whole_ct = df.count()
sample_ct = df.sample(False, 0.2, 2026).count()
print(sample_ct / whole_ct)

In [ ]:
# Random splits
dataframes = df.randomSplit([0.8, 0.2], seed=2026)
print(dataframes[0].count() > dataframes[1].count())

In [ ]:
spark.stop()